#### Проверка работоспособности через requests

In [3]:
import requests

payload = {
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "messages": [
        {"role": "user", "content": "What is the capital of Germany?"}
    ],
    "temperature": 0.0,
    "max_tokens": 64
}

r = requests.post(
    "http://localhost:8001/v1/chat/completions",
    json=payload
)

print(r.json()["choices"][0]["message"]["content"])


The capital of Germany is Berlin.


#### Проверка работоспособности через openai

In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="EMPTY"
)

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[
        {"role": "user", "content": "What is the capital of Germany?"}
    ],
    temperature=0.0
)

print(resp.choices[0].message.content)


The capital of Germany is Berlin.


In [ ]:
def llm_answer(query):
    

#### LLM as a judge

In [8]:
import pandas as pd

In [64]:
# Прогоняем модель и генерируем ответы в столбец model_answer
df = pd.read_csv("qa_answers.csv")
df = df.rename(columns = {'Вопрос': 'question', 'Правильный ответ': 'reference_answer'})

In [65]:
ans = []
for row in df.iterrows():
    query = row[-1]['question']
    resp = client.chat.completions.create(
        model="Qwen/Qwen2.5-3B-Instruct",
        messages=[
            {"role": "system", "content": "Ты полезный помощник, отвечай кратко!"},
            {"role": "user", "content": query}
        ],
        temperature=0.1,
        max_tokens=32,
    )
    ans.append(resp.choices[0].message.content)

In [66]:
df['model_answer'] = ans

In [67]:
df.head()

,question,reference_answer,model_answer
0,Кто первым человеком высадился на Луну?,Нил Армстронг,"Первым человеком, который высадился на Луну, б..."
1,Какую теорию разработал Альберт Эйнштейн?,Теорию относительности,Альберт Эйнштейн разработал специальную и обоб...
2,Кто написал роман '1984'?,Джордж Оруэлл,"Роман ""1984"" написал Орwell (Бернард Оруэлл)."
3,Кто был премьер-министром Великобритании во вр...,Уинстон Черчилль,Либертарианский премьер-министр Уинстон Черчил...
4,Кто исполнил песню 'Thriller'?,Майкл Джексон,"Песня ""Thriller"" была исполнена группой The Ja..."


In [ ]:
### Создаем кастомную метрику и прогоняем эксперимент

In [22]:
GRADING_PROMPT = '''
Вы - беспристрастный судья, оценивающий ответ.

Вопрос:
{question}

Примерный ответ:
{model_answer}

Эталонный ответ:
{reference_answer}

Оцените примерный ответ по шкале от 1 до 5:
1 - полностью неверный
3 - частично правильный
5 - полностью корректно и точно

Возвращает ТОЛЬКО одно целое число от 1 до 5.
'''

In [68]:
### Создаем метрику
def judge_score(question, model_answer, reference_answer):
    #score = call_judge_llm(question, model_answer, reference_answer)
    resp = client.chat.completions.create(
        model="Qwen/Qwen2.5-3B-Instruct",
        messages=[
            {"role": "system", "content": "Ты полезный помощник, отвечай кратко!"},
            {"role": "user", "content": GRADING_PROMPT.format(
                    question = question,
                    model_answer = model_answer,
                    reference_answer = reference_answer,
                )
            }
        ],
        temperature=0.0,
        max_tokens=5,
    )
    score = resp.choices[0].message.content
    return float(score)


In [72]:
question = df.iloc[0]['question']
model_answer = df.iloc[0]['model_answer']
reference_answer = df.iloc[0]['reference_answer']
print(
    'question:', question,
    '\nmodel_answer:', model_answer,
    '\nreference_answer:', reference_answer
)
print(judge_score(question, model_answer, reference_answer))

question: Кто первым человеком высадился на Луну? 
model_answer: Первым человеком, который высадился на Луну, был Аполлон-11. 
reference_answer: Нил Армстронг
3.0


In [23]:
from mlflow.metrics.genai import make_genai_metric

judge_metric = make_genai_metric(
    name="answer_correctness",
    definition="Factual correctness of model answer.",
    grading_prompt=GRADING_PROMPT,
    model="openai:/Qwen/Qwen2.5-3B-Instruct",
    parameters={
        "temperature": 0.0,
        "max_tokens": 5
    }
)


In [25]:
import os

os.environ["MLFLOW_DEPLOYMENTS_TARGET"] = "openai"
os.environ["OPENAI_API_BASE"] = "http://localhost:8001/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"

In [28]:
MLFLOW_URI = "http://194.156.124.212:8501"

In [73]:
import mlflow
import pandas as pd
import numpy as np

mlflow.set_experiment("llm_as_judge_qwen")

with mlflow.start_run(run_name="llm_judge_run"):
    scores = []

    for _, row in df.iterrows():
        s = judge_score(
            row["question"],
            row["model_answer"],
            row["reference_answer"]
        )
        scores.append(s)

    mlflow.log_metric("judge_mean", np.mean(scores))
    mlflow.log_metric("judge_std", np.std(scores))
    mlflow.log_metric("judge_min", np.min(scores))
    mlflow.log_metric("judge_max", np.max(scores))


🏃 View run llm_judge_run at: http://194.156.124.212:8501/#/experiments/684744205207162000/runs/c5f1f8802e6141d5a79c356ae00b23b8
🧪 View experiment at: http://194.156.124.212:8501/#/experiments/684744205207162000
